# 04 - 端到端流程（冒烟 / 全量 + 断点续跑）

本 Notebook 是 **`docs/user_guide.md`** 中命令行流水线的交互版，支持 **marker 断点续跑**（日志在 `runs/pipeline-logs/`）。

## 两套测试流程

| 流程 | 对应脚本 | 实验 YAML | 说明 |
|------|----------|-----------|------|
| **冒烟测试** | `build_index.sh smoke` + `exp_baseline` 各步 | `exp_baseline.yaml` | ~500 行索引、`mcelog_max_rows`/`mcelog_scan_rows` 均为 **800000**（须一致）；**overrides 关闭 DDP** 以加速；preprocess 已含 validate |
| **全量测试** | `build_index.sh full` + `exp_full_best_ddp` | `exp_full_best_ddp.yaml` | 取消行数截断；**系统 RAM 预算 50%**；**gpu/ddp 继承 base 默认多卡** |

**推荐顺序**：先跑完 **冒烟** 单元确认环境 → 再按需跑 **全量**（耗时长、占 GPU/磁盘）。

## 产物位置

- 训练/推理：`runs/<run_name>/<时间戳>/`（`checkpoints/grid_cnn.pt`、`predictions.csv` 等）
- 会话指针：`runs/<run_name>/.latest_train_session` / `.latest_infer_session`
- 流水线日志：`runs/pipeline-logs/<exp>__<step>__*.log`

## 与其他 Notebook

- `01_eda.ipynb`：raw 表概览（冒烟/全量前可先确认文件存在）
- `02_feature_analysis.ipynb`：preprocess 后看 `X` 形状与分布
- `03_error_analysis.ipynb`：infer 后看 FP/FN

## 使用提示

- **中断恢复**：重新运行对应流程单元格；已完成步骤因 marker 存在而跳过。
- **从头重跑某流程**：删除 `runs/pipeline-logs/markers/` 下对应 `smoke__*` 或 `full_best__*` 的 `.done` 文件，或运行文末「重置 marker」。
- **改 horizon / 索引 / 网格**：清空 `data/processed`、`data/splits` 并删除 preprocess marker 后重跑。
- **同配置重跑 preprocess**：可直接再跑（split 会自动覆盖）；Notebook 内旧输出日志格式可能与当前代码略有差异。
- 异常时 **Kernel → Restart** 再跑（避免缓存旧版 `dram2cnn`）。

命令行等价：冒烟 **`bash scripts/autotune.sh`**（preprocess 含 validate → train → infer → evaluate，不含 build_index）；全量见 **`docs/user_guide.md` §6**。

In [ ]:
from pathlib import Path
from datetime import datetime
import os

# 从项目根或 notebooks/ 打开均可
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT / 'src'}:{os.environ.get('PYTHONPATH', '')}"

LOG_DIR = PROJECT_ROOT / "runs" / "pipeline-logs"
MARKER_DIR = LOG_DIR / "markers"
LOG_DIR.mkdir(parents=True, exist_ok=True)
MARKER_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOG_DIR:", LOG_DIR)
print("MARKER_DIR:", MARKER_DIR)

In [ ]:
import subprocess

_SRC = PROJECT_ROOT / "src"


def run_step(exp_name: str, step_name: str, command: str) -> None:
    """执行一步 shell 命令；成功则写 marker，便于断点续跑。"""
    marker = MARKER_DIR / f"{exp_name}__{step_name}.done"
    if marker.exists():
        print(f"[SKIP] {exp_name}/{step_name} (marker: {marker.name})")
        return

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = LOG_DIR / f"{exp_name}__{step_name}__{ts}.log"
    cmd = (
        f'bash -lc "set -o pipefail; export PYTHONUNBUFFERED=1; '
        f'export PYTHONPATH={_SRC}:${{PYTHONPATH:-}}; '
        f'{command} 2>&1 | tee \"{log_file}\""'
    )

    print(f"[RUN ] {exp_name}/{step_name}")
    print("      command:", command)
    print("      log:", log_file)
    subprocess.run(cmd, shell=True, check=True, cwd=str(PROJECT_ROOT))
    marker.write_text(ts, encoding="utf-8")
    print(f"[DONE] {exp_name}/{step_name}\n")


def clear_exp_markers(exp_name: str) -> int:
    """删除某一实验（如 smoke / full_best）的全部 marker，便于该流程从头重跑。"""
    removed = 0
    for p in sorted(MARKER_DIR.glob(f"{exp_name}__*.done")):
        p.unlink(missing_ok=True)
        removed += 1
    print(f"已删除 {exp_name} 的 marker 数量: {removed}")
    return removed

In [ ]:
# 通用预检：切分形状与标签（建议在 train 前、冒烟/全量均可运行）
import numpy as np

try:
    import yaml
except ImportError:
    yaml = None


def preflight_splits_vs_model(root: Path | None = None) -> None:
    root = root or PROJECT_ROOT
    model_p = root / "configs/model.yaml"
    x_train = root / "data/splits/X_train.npy"
    if yaml is None or not model_p.is_file():
        print("[preflight] 跳过 model.yaml 形状检查")
        return
    if not x_train.is_file():
        print("[preflight] 尚无 X_train.npy，请先 preprocess")
        return
    cfg = yaml.safe_load(model_p.read_text(encoding="utf-8"))
    exp_c, exp_h, exp_w = int(cfg["in_channels"]), int(cfg["grid_height"]), int(cfg["grid_width"])
    x = np.load(x_train, mmap_mode="r")
    _n, c, h, w = (int(x.shape[i]) for i in range(4))
    if (c, h, w) == (exp_c, exp_h, exp_w):
        print(f"[preflight] 通过：X_train (C,H,W)=({c},{h},{w})")
    else:
        print(f"[preflight] 不匹配：X_train=({c},{h},{w}) vs model=({exp_c},{exp_h},{exp_w})；删 preprocess marker 后重跑")


def preflight_split_labels(root: Path | None = None) -> None:
    root = root or PROJECT_ROOT
    for tag in ("train", "val"):
        p = root / "data/splits" / f"y_{tag}.npy"
        if not p.is_file():
            continue
        y = np.load(p)
        pos, n = int(y.sum()), int(len(y))
        if pos == 0:
            print(f"[preflight] 警告：{tag} 无正样本 → AUC/PR 可能为 nan")
        else:
            print(f"[preflight] {tag}: N={n} pos={pos} rate={pos / max(n, 1):.4f}")


preflight_splits_vs_model(PROJECT_ROOT)
preflight_split_labels(PROJECT_ROOT)

---
## 冒烟测试（Smoke）

目标：快速验证 **索引 → 预处理 → 训练 → 推理 → 评估** 全链路可用。

| 步骤 | 说明 |
|------|------|
| 1. 生成演示索引 | `features_5m_index_demo.csv`（~500 行） |
| 2. preprocess | `exp_baseline.yaml`，截断 mcelog/索引 |
| 3. train | `train.sh`（exp_baseline **关闭 DDP** 以加速） |
| 4. infer / evaluate | 推理与指标 |

命令行一键等价（无 marker）：**`bash scripts/autotune.sh`** = preprocess（含 validate）→ train → infer → evaluate（**不含** build_index/inspect_data；完整冒烟见本 Notebook 冒烟单元）。

In [ ]:
# 【冒烟 1/2】重建演示索引（与 configs/data.yaml 默认 index_features_path 一致）
SMOKE_EXP = "smoke"
run_step(SMOKE_EXP, "build_index", "bash scripts/build_index.sh smoke")

In [ ]:
# 【冒烟 2/2】baseline 全流程（configs/experiments/exp_baseline.yaml）
SMOKE_CFG = "configs/experiments/exp_baseline.yaml"

run_step(SMOKE_EXP, "preprocess", f"bash scripts/preprocess.sh {SMOKE_CFG}")
# preprocess.sh 末尾已 validate；以下为可选重复诊断
# run_step(SMOKE_EXP, "validate", f"bash scripts/validate.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "inspect_data", f"bash scripts/inspect_data.sh {SMOKE_CFG}")

preflight_splits_vs_model(PROJECT_ROOT)
preflight_split_labels(PROJECT_ROOT)

run_step(SMOKE_EXP, "train", f"bash scripts/train.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "infer", f"bash scripts/infer.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "evaluate", f"bash scripts/evaluate.sh {SMOKE_CFG}")

print("冒烟完成 → runs/exp_tianchi_baseline/")
print("后续分析：02_feature_analysis.ipynb（X 分布）、03_error_analysis.ipynb（FP/FN，EXP 仍用 exp_baseline）")

In [ ]:
# 【可选】命令行 autotune 一键冒烟（不分 marker；与上一单元二选一即可）
# run_step(SMOKE_EXP, "autotune", "bash scripts/autotune.sh")

---
## 全量测试（Full + DDP）

目标：在 **真实全量索引 + 完整 mcelog** 上训练（多卡 DDP）。

| 步骤 | 说明 |
|------|------|
| 1. 生成全量索引 | 输出 `data/raw/features_index_h300s_step300s_full.csv`（耗时长） |
| 2. **配置** | `exp_full_best_ddp.yaml` 已含全量索引；系统 **RAM 峰值 ≤50%**（`data.yaml` `ram_budget_frac: 0.5`） |
| 3. 清空旧产物 | 删除 `data/processed/*`、`data/splits/*` 及 `full_best__preprocess.done` |
| 4. preprocess | **建议在终端**跑；日志应含「目标≤50%」、worker 通常=1 |
| 5. train_ddp / infer / evaluate | preprocess 完成后 **Kernel Restart**，再跑后续单元 |

> **注意**：仅取消 `max_index_rows` 而不改 `index_features_path` 时，仍会用 ~500 行演示索引。  
> 终端等价：`bash scripts/preprocess.sh configs/experiments/exp_full_best_ddp.yaml`

In [ ]:
# 【全量 1/3】扫描完整 mcelog 生成全量索引（耗时长，可跳过若 CSV 已存在且 marker 在）
FULL_EXP = "full_best"
run_step(FULL_EXP, "build_index", "bash scripts/build_index.sh full")

print(
    "\n下一步：\n"
    "  1) exp_full_best_ddp.yaml 已默认 index_features_path=data/raw/features_index_h300s_step300s_full.csv\n"
    "  2) 若曾跑过冒烟 preprocess：rm -rf data/processed/* data/splits/*\n"
    "  3) clear_exp_markers('full_best')  # 或删 full_best__preprocess.done\n"
    "  4) 运行下方「全量预检 + 流水线」单元（preprocess 建议终端执行）"
)

In [ ]:
# 【全量 2/3】预检：index_features_path 不得仍指向演示 stub
import sys

_src = str(PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from dram2cnn.utils.config import load_experiment_config
from dram2cnn.utils.experiment_guards import assert_full_best_index_not_stub

FULL_CFG = "configs/experiments/exp_full_best_ddp.yaml"
_cfg = load_experiment_config(PROJECT_ROOT / FULL_CFG)
assert_full_best_index_not_stub(_cfg, context="04_full_best_preflight")
print(f"[preflight] 全量 index_features_path OK: {_cfg['data']['index_features_path']}")

In [ ]:
# 【全量 3/3】exp_full_best_ddp：preprocess → DDP 训练 → infer → evaluate
run_step(FULL_EXP, "preprocess", f"bash scripts/preprocess.sh {FULL_CFG}")
# run_step(FULL_EXP, "validate", f"bash scripts/validate.sh {FULL_CFG}")
run_step(FULL_EXP, "inspect_data", f"bash scripts/inspect_data.sh {FULL_CFG}")

preflight_splits_vs_model(PROJECT_ROOT)
preflight_split_labels(PROJECT_ROOT)

# 多卡：运行前在终端 export CUDA_VISIBLE_DEVICES=0,1,2
run_step(FULL_EXP, "train", f"bash scripts/train_ddp.sh {FULL_CFG}")
run_step(FULL_EXP, "infer", f"bash scripts/infer.sh {FULL_CFG}")
run_step(FULL_EXP, "evaluate", f"bash scripts/evaluate.sh {FULL_CFG}")

print("全量完成 → runs/exp_tianchi_full_best_ddp/")
print("分析 03 时请将 EXP_YAML 改为 exp_full_best_ddp.yaml，或手动指定 predictions 路径")

In [ ]:
# 查看最近日志与 marker
print("\n=== 最近日志（最多 15 条）===")
for p in sorted(LOG_DIR.glob("*.log"), key=lambda x: x.stat().st_mtime, reverse=True)[:15]:
    print(p.name)

print("\n=== 已完成 marker ===")
for p in sorted(MARKER_DIR.glob("*.done")):
    print(p.name, "=>", p.read_text(encoding="utf-8").strip())

In [ ]:
# 重置 marker（可选 — 取消注释后再运行本格）
# clear_exp_markers("smoke")      # 仅重跑冒烟
# clear_exp_markers("full_best")  # 仅重跑全量
#
# CLEAR_ALL_MARKERS = False
# if CLEAR_ALL_MARKERS:
#     removed = 0
#     for p in sorted(MARKER_DIR.glob("*.done")):
#         p.unlink(missing_ok=True)
#         removed += 1
#     print(f"已删除全部 marker 数量: {removed}")
# else:
print("未删除 marker。需重跑某流程时取消上方 clear_exp_markers(...) 注释。")